# Rerun of the model

I will run the baseline model to check that all functions also work on real data
We should obtain the same results

In [51]:
# Imports

import pandas as pd
import numpy as np 
from vportfolio import engine, allocators, metrics, trials
from vportfolio.params import RAW, PROCESSED, LOOKBACK, CAP, RESULTS, BACKTEST, TRIALS_LOG
import matplotlib.pyplot as plt
import os

In [52]:
# load the data

raw_data = pd.read_parquet(RAW / "yahoo_prices_2026-10-23.parquet")
rebal_dates = pd.read_parquet(PROCESSED / "rebal_dates.parquet")["date"]
cost_bp = pd.read_parquet(PROCESSED / "cost_bp.parquet")["cost_bp"]
rf_daily = pd.read_parquet(PROCESSED / "rf_daily.parquet")["rf"]

In [53]:
# variables definition

b = ["SPY", "AGG"]
close_b = raw_data["Close"][b]
open_b  = raw_data["Open"][b]
eligible_b = close_b.notna() # ETFs that have traded
returns_b = close_b.pct_change(fill_method=None)

In [54]:
# 60/40
result_6040 = engine.backtest_engine(allocators.sixty_forty, close_b, open_b, returns_b, eligible_b, rebal_dates, cost_bp[["SPY", "AGG"]], cap=None, lookback = 1)

In [55]:
NAV_6040 = result_6040["NAV"]
turnover_6040 = result_6040["Turnover"]
performance_6040 = metrics.performance_calculator(NAV_6040, rf_daily)
return_6040 = performance_6040["ann_return"]
sharpe_6040 = performance_6040["sharpe"]
drawdow_6040 = performance_6040["max_drawdown"]

In [56]:
# load the data

close_ew = pd.read_parquet(PROCESSED / "close.parquet")
open_ew = pd.read_parquet(PROCESSED / "open.parquet")
returns_ew = pd.read_parquet(PROCESSED / "returns.parquet")
eligible_ew = pd.read_parquet(PROCESSED / "eligible.parquet")

In [57]:
# equal weights
result_ew = engine.backtest_engine(
    allocators.equal_weights, close_ew, open_ew, returns_ew, eligible_ew, rebal_dates, cost_bp, 
    cap=CAP, lookback = LOOKBACK)

In [58]:
# metrics
NAV_ew = result_ew["NAV"]
turnover_ew = result_ew["Turnover"]
performance_ew = metrics.performance_calculator(NAV_ew, rf_daily)
return_ew = performance_ew["ann_return"]
sharpe_ew = performance_ew["sharpe"]
drawdow_ew = performance_ew["max_drawdown"]

In [59]:
# risk weighted
result_rw = engine.backtest_engine(
    allocators.risk_weighted, close_ew, open_ew, returns_ew, eligible_ew, rebal_dates, cost_bp, 
    cap=CAP, lookback = LOOKBACK)

In [60]:
# metrics
NAV_rw = result_rw["NAV"]
turnover_rw = result_rw["Turnover"]
performance_rw = metrics.performance_calculator(NAV_rw, rf_daily)
return_rw = performance_rw["ann_return"]
sharpe_rw = performance_rw["sharpe"]
drawdow_rw = performance_rw["max_drawdown"]

In [61]:
# we now load the old log to doublecheck
old = pd.read_csv(TRIALS_LOG).set_index("name")

In [62]:
# we create the new log and compare to the old one
new = pd.Series({"sixty_forty": performance_6040["sharpe"],
                 "equal_weights": performance_ew["sharpe"],
                 "risk_weighted": performance_rw["sharpe"]})
assert np.isclose(old.loc[new.index, "sharpe"], new, atol=1e-12).all()
print("the logs are identical: test suceeded")

the logs are identical: test suceeded


In [63]:
# save the new parquet
NAVs = {"sixty_forty": NAV_6040, "equal_weights": NAV_ew, "risk_weighted": NAV_rw}
paths = {name: trials.save_returns(name, nav) for name, nav in NAVs.items()}

In [65]:
# build new rows for the logs

row_6040 = trials.make_new_row("sixty_forty", allocators.sixty_forty, "benchmark",
                               performance_6040, turnover_6040, paths["sixty_forty"],
                               cap=None, lookback=1)
row_ew = trials.make_new_row("equal_weights", allocators.equal_weights, "benchmark",
                             performance_ew, turnover_ew, paths["equal_weights"])
row_rw = trials.make_new_row("risk_weighted", allocators.risk_weighted, "diagnostic",
                             performance_rw, turnover_rw, paths["risk_weighted"])

In [66]:
trials.update_trials_log(pd.DataFrame([row_6040, row_ew, row_rw]))

,date,name,allocator,settings,role,ann_return,sharpe,turnover,max_drawdown,returns_file,ann_vol,avg_turnover
0,2026-10-29,sixty_forty,sixty_forty,"cap=None, lookback=1",benchmark,0.082404,0.649639,NaN,-0.356924,/home/antonio/code/aferri-git/ML_virtual_portf...,0.114335,0.021518
1,2026-10-29,equal_weights,equal_weights,"cap=0.2, lookback=504",benchmark,0.057396,0.492381,NaN,-0.337178,/home/antonio/code/aferri-git/ML_virtual_portf...,0.100348,0.037566
2,2026-10-29,risk_weighted,risk_weighted,"cap=0.2, lookback=504",diagnostic,0.041755,0.587724,NaN,-0.194149,/home/antonio/code/aferri-git/ML_virtual_portf...,0.052478,0.034572


In [ ]:
# date is today 29th of october so the logs are updated succesfully